In [1]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
model = AutoModelForCausalLM.from_pretrained("distilgpt2").to(device)
tokenizer = AutoTokenizer.from_pretrained("distilgpt2")

model.eval()
print("model is on:", next(model.parameters()).device)

config.json:   0%|          | 0.00/762 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  353MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/76 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model is on: cuda:0


In [2]:
from datasets import load_dataset

test_data = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", split = "test")

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-2-raw-v1/test-00000-of-00001.pa(…): reconstructing file:   0%|          |  0.00B /  733kB            

wikitext-2-raw-v1/test-00000-of-00001.pa(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/train-00000-of-00001.p(…): reconstructing file:   0%|          |  0.00B / 6.36MB            

wikitext-2-raw-v1/train-00000-of-00001.p(…): downloading bytes:           |  0.00B            

wikitext-2-raw-v1/validation-00000-of-00(…): reconstructing file:   0%|          |  0.00B /  657kB            

wikitext-2-raw-v1/validation-00000-of-00(…): downloading bytes:           |  0.00B            

Generating test split:   0%|          | 0/4358 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/36718 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/3760 [00:00<?, ? examples/s]

In [3]:
def evaluate_perplexity(model, dataset, tokenizer, device, max_len = 1024, stride = 512):

  text = "\n\n".join(dataset["text"])
  encodings = tokenizer(text, return_tensors = "pt")
  input_ids_all = encodings.input_ids.to(device)
  seq_len = input_ids_all.size(1)

  prev_end = 0
  nll_sum = 0
  n_tokens = 0

  for begin in range(0, seq_len, stride):
    end = min(begin + max_len, seq_len)
    target_len = end - prev_end

    input_ids = input_ids_all[:, begin:end]
    target_ids = input_ids.clone()
    target_ids[:, :-target_len] = -100

    with torch.no_grad():
      output = model(input_ids, labels = target_ids)
      neg_log_loss = output.loss

    nll_sum += neg_log_loss * target_len
    n_tokens += target_len
    prev_end = end
    if end == seq_len:
      break

  avg_nll = nll_sum / n_tokens
  perplexity = torch.exp(avg_nll)
  return perplexity.item()

In [4]:
baseline_ppl = evaluate_perplexity(model, test_data, tokenizer, device)
print("The Baseline perplexity for FP32 Precision: ", baseline_ppl)

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (287644 > 1024). Running this sequence through the model will result in indexing errors
[transformers] `loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


The Baseline perplexity for FP32 Precision:  39.25837707519531


In [5]:
def analyse_model_size(model):
  quantize_bytes = 0
  total_bytes = 0
  skip_bytes = 0

  targets = ("c_attn", "c_proj", "c_fc")

  for name, param in model.named_parameters():
    param_bytes = param.numel() * param.element_size()
    total_bytes += param_bytes

    is_weight = name.endswith(".weight")
    is_target = any(t in name for t in targets)

    if is_weight and is_target:
      quantize_bytes += param_bytes
      bucket = "QUANTIZE"
    else:
      skip_bytes += param_bytes
      bucket = "skip"

    mb = param_bytes / 1e6
    print(f"{bucket:9} {mb:6.2f}MB  {tuple(param.shape)!s:18} {name}")

  return total_bytes, quantize_bytes, skip_bytes

total, quant, skip = analyse_model_size(model)
print(f"\nTotal model size:      {total/1e6:7.2f} MB")
print(f"Weights we quantize:   {quant/1e6:7.2f} MB  ({100*quant/total:.1f}% of total)")
print(f"Weights we skip:       {skip/1e6:7.2f} MB  ({100*skip/total:.1f}% of total)")

skip      154.39MB  (50257, 768)       transformer.wte.weight
skip        3.15MB  (1024, 768)        transformer.wpe.weight
skip        0.00MB  (768,)             transformer.h.0.ln_1.weight
skip        0.00MB  (768,)             transformer.h.0.ln_1.bias
QUANTIZE    7.08MB  (768, 2304)        transformer.h.0.attn.c_attn.weight
skip        0.01MB  (2304,)            transformer.h.0.attn.c_attn.bias
QUANTIZE    2.36MB  (768, 768)         transformer.h.0.attn.c_proj.weight
skip        0.00MB  (768,)             transformer.h.0.attn.c_proj.bias
skip        0.00MB  (768,)             transformer.h.0.ln_2.weight
skip        0.00MB  (768,)             transformer.h.0.ln_2.bias
QUANTIZE    9.44MB  (768, 3072)        transformer.h.0.mlp.c_fc.weight
skip        0.01MB  (3072,)            transformer.h.0.mlp.c_fc.bias
QUANTIZE    9.44MB  (3072, 768)        transformer.h.0.mlp.c_proj.weight
skip        0.00MB  (768,)             transformer.h.0.mlp.c_proj.bias
skip        0.00MB  (768,)          

In [6]:
def quantize_dequantize_per_tensor(W):

  max_abs = W.abs().max()
  Scale = max_abs / 127.0

  W_quant = torch.round(W / Scale).clamp(-127,127)
  W_dequant = W_quant * Scale

  return W_dequant

In [7]:
import copy

def quantize_model(model, quant_fn):

  q_model = copy.deepcopy(model)
  targets = ("c_attn", "c_proj", "c_fc")

  with torch.no_grad():
    for name, param in q_model.named_parameters():
      is_weight = name.endswith(".weight")
      is_target = any(t in name for t in targets)
      if is_weight and is_target:
        param.copy_(quant_fn(param.data))

  return q_model

In [8]:
qmodel_per_tensor = quantize_model(model, quantize_dequantize_per_tensor)

ppl_per_tensor = evaluate_perplexity(qmodel_per_tensor, test_data, tokenizer, device)

print(f"FP32 baseline perplexity:      {baseline_ppl:6.2f}")
print(f"INT8 per-tensor perplexity:    {ppl_per_tensor:6.2f}")
print(f"Change:                        +{ppl_per_tensor - baseline_ppl:.2f}")

FP32 baseline perplexity:       39.26
INT8 per-tensor perplexity:     50.05
Change:                        +10.79


In [9]:
def quantize_dequantize_per_channel(W):

  max_abs = W.abs().max(dim = 0, keepdim = True).values
  Scale = max_abs / 127.0

  W_quant = torch.round(W / Scale).clamp(-127,127)
  W_dequant = W_quant * Scale

  return W_dequant

In [10]:
sample = model.transformer.h[0].mlp.c_fc.weight.data
pt = quantize_dequantize_per_tensor(sample)
pc = quantize_dequantize_per_channel(sample)

print("Original:      ", sample.flatten()[:5])
print("Per-tensor:    ", pt.flatten()[:5])
print("Per-channel:   ", pc.flatten()[:5])
print("\nPer-tensor  MAX error:", (sample-pt).abs().max().item())
print("Per-channel MAX error:", (sample-pc).abs().max().item())
print("\nPer-tensor  MEAN error:", (sample-pt).abs().mean().item())
print("Per-channel MEAN error:", (sample-pc).abs().mean().item())

Original:       tensor([ 0.1239,  0.1034, -0.0096,  0.0898, -0.1020], device='cuda:0')
Per-tensor:     tensor([ 0.1241,  0.0828, -0.0000,  0.0828, -0.0828], device='cuda:0')
Per-channel:    tensor([ 0.1228,  0.1042, -0.0113,  0.0897, -0.1028], device='cuda:0')

Per-tensor  MAX error: 0.020688191056251526
Per-channel MAX error: 0.020653652027249336

Per-tensor  MEAN error: 0.010344254784286022
Per-channel MEAN error: 0.0010896018939092755


In [11]:
qmodel_per_channel = quantize_model(model, quantize_dequantize_per_channel)

ppl_per_channel = evaluate_perplexity(qmodel_per_channel, test_data, tokenizer, device)

print(f"FP32 baseline perplexity:      {baseline_ppl:6.2f}")
print(f"INT8 per-tensor perplexity:    {ppl_per_tensor:6.2f}   (+{ppl_per_tensor - baseline_ppl:.2f})")
print(f"INT8 per-channel perplexity:   {ppl_per_channel:6.2f}   (+{ppl_per_channel - baseline_ppl:.2f})")

FP32 baseline perplexity:       39.26
INT8 per-tensor perplexity:     50.05   (+10.79)
INT8 per-channel perplexity:    39.33   (+0.08)


In [12]:
def quantize_dequantize_int4_per_group(W, group_size = 64):
  in_features, out_features = W.shape
  W_out = torch.empty_like(W)

  for start in range(0, in_features, group_size):
    end = min(start + group_size, in_features)
    group = W[start : end, :]

    max_abs = group.abs().max(dim = 0, keepdim = True).values
    Scale = max_abs / 7.0

    group_quant = torch.round(group / Scale).clamp(-7,7)
    W_out[start:end, :] = group_quant * Scale

  return W_out

In [13]:
W = model.transformer.h[0].mlp.c_fc.weight.data

pc  = quantize_dequantize_per_channel(W)
pg = quantize_dequantize_int4_per_group(W, 64)

print("INT8 per-channel   MEAN error:", (W-pc).abs().mean().item())
print("INT4 group-wise    MEAN error:", (W-pg).abs().mean().item())

INT8 per-channel   MEAN error: 0.0010896018939092755
INT4 group-wise    MEAN error: 0.013849890790879726


In [14]:
def pack_int4(values_signed):
  v = (values_signed + 8).to(torch.uint8)
  low = v[0::2]
  high = v[1::2]

  packed = (high << 4) | low
  return packed

def unpack_int4(packed):
  low = packed & 0x0F
  high = packed >> 4
  v = torch.empty(packed.numel() * 2, dtype = torch.uint8, device = packed.device)
  v[0::2] = low
  v[1::2] = high
  return v.to(torch.int16) - 8

In [16]:
def int4_per_group_storage_bytes(model, group_size=64):
    targets = ("c_attn", "c_proj", "c_fc")
    packed_weight_bytes = 0
    scale_bytes = 0

    for name, param in model.named_parameters():
        is_weight = name.endswith(".weight")
        is_target = any(t in name for t in targets)
        if is_weight and is_target:
            in_f, out_f = param.shape
            packed_weight_bytes += (param.numel() + 1) // 2
            n_groups = (in_f + group_size - 1) // group_size
            scale_bytes += n_groups * out_f * 4

    return packed_weight_bytes, scale_bytes

pw, sc = int4_per_group_storage_bytes(model, 64)
int4_total = pw + sc

fp32_quant_layers = quant
int8_quant_layers = quant / 4

print(f"Quantized layers, FP32:   {fp32_quant_layers/1e6:7.2f} MB")
print(f"Quantized layers, INT8:   {int8_quant_layers/1e6:7.2f} MB")
print(f"INT4 packed weights:      {pw/1e6:7.2f} MB")
print(f"INT4 group scales:        {sc/1e6:7.2f} MB")
print(f"INT4 total:               {int4_total/1e6:7.2f} MB")
print(f"INT4 vs FP32:             {fp32_quant_layers/int4_total:.2f}x smaller")

Quantized layers, FP32:    169.87 MB
Quantized layers, INT8:     42.47 MB
INT4 packed weights:        21.23 MB
INT4 group scales:           2.65 MB
INT4 total:                 23.89 MB
INT4 vs FP32:             7.11x smaller


In [17]:
qmodel_int4 = quantize_model(model, quantize_dequantize_int4_per_group)

ppl_int4 = evaluate_perplexity(qmodel_int4, test_data, tokenizer, device)

print(f"FP32 baseline:            {baseline_ppl:.2f}")
print(f"INT8 per-tensor:          {ppl_per_tensor:.2f}   (+{ppl_per_tensor - baseline_ppl:.2f})")
print(f"INT8 per-channel:         {ppl_per_channel:.2f}   (+{ppl_per_channel - baseline_ppl:.2f})")
print(f"INT4 group-wise (group_size = 64):   {ppl_int4:.2f}   (+{ppl_int4 - baseline_ppl:.2f})")

FP32 baseline:            39.26
INT8 per-tensor:          50.05   (+10.79)
INT8 per-channel:         39.33   (+0.08)
INT4 group-wise (group_size = 64):   48.01   (+8.75)
